# Original TypiClust Algorithm
Reproduction of TPC_RP (SimCLR + K-Means + Typicality Selection) on CIFAR-10 with B=10.

**Note:** The SimCLR encoder (ResNet-18, 200 epochs) is trained from scratch in a separate notebook (`SimCLR_implementation.ipynb`). The training is split out so the algorithm can be run without retraining each time. The saved weights are loaded below.

## 1. Setup & Data Loading

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
# Load CIFAR-10
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=256, shuffle=False, num_workers=2)
testloader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

## 2. SimCLR Encoder (Representation Learning)

In [3]:
class ResNet18Encoder(nn.Module):
    """SimCLR encoder with ResNet-18 backbone adapted for CIFAR-10 (32x32)."""
    def __init__(self):
        super().__init__()
        self.resnet = models.resnet18(weights=None)
        self.resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.resnet.maxpool = nn.Identity()
        self.resnet.fc = nn.Identity()

    def forward(self, x):
        return self.resnet(x)


def load_encoder(weights_path, device='cuda'):
    encoder = ResNet18Encoder()
    encoder.load_state_dict(torch.load(weights_path, map_location=device))
    encoder = encoder.to(device)
    encoder.eval()
    return encoder


def extract_embeddings(encoder, dataloader, device='cuda'):
    encoder.eval()
    all_embeddings = []
    all_labels = []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            h = encoder(images)
            all_embeddings.append(h.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.vstack(all_embeddings), np.hstack(all_labels)

In [4]:
# Load pretrained SimCLR encoder and extract embeddings
encoder = load_encoder('./simclr_encoder_200ep.pth')
embeddings, labels = extract_embeddings(encoder, trainloader)
test_embeddings, test_labels = extract_embeddings(encoder, testloader)
print(f"Train embeddings: {embeddings.shape}")
print(f"Test embeddings: {test_embeddings.shape}")

Train embeddings: (50000, 512)
Test embeddings: (10000, 512)


## 3. Typicality Computation

In [5]:
def compute_typicality(embeddings, k=20):
    """Compute typicality as inverse mean distance to k nearest neighbours."""
    nn = NearestNeighbors(n_neighbors=k + 1)
    nn.fit(embeddings)
    distances, _ = nn.kneighbors(embeddings)
    distances = distances[:, 1:]  # exclude self
    return np.mean(distances, axis=1) ** -1


typicality = compute_typicality(embeddings, k=20)
print(f"Typicality shape: {typicality.shape}")
print(f"Typicality range: [{typicality.min():.4f}, {typicality.max():.4f}]")

Typicality shape: (50000,)
Typicality range: [0.0637, 0.4876]


## 4. Clustering (K-Means, K=B)

In [6]:
def cluster_standard(embeddings, budget, random_state=42):
    """K-Means clustering with K = budget."""
    kmeans = KMeans(n_clusters=budget, random_state=random_state, n_init=10)
    kmeans.fit(embeddings)
    return kmeans.labels_


budget = 10
cluster_labels = cluster_standard(embeddings, budget)
print(f"Cluster sizes: {np.bincount(cluster_labels)}")

Cluster sizes: [4518 4837 9152 4655 6773 5676 4402 4942 3035 2010]


## 5. Selection (Max Typicality)

In [7]:
def select_max_typicality(typicality, cluster_labels, budget):
    """Select the most typical point from each cluster."""
    selected = []
    for cluster in range(budget):
        cluster_idx = np.where(cluster_labels == cluster)[0]
        best = cluster_idx[np.argmax(typicality[cluster_idx])]
        selected.append(best)
    return np.array(selected)


selected_indices = select_max_typicality(typicality, cluster_labels, budget)
print(f"Selected indices: {selected_indices}")
print(f"Selected labels: {labels[selected_indices]}")
print(f"Classes covered: {len(np.unique(labels[selected_indices]))}")

Selected indices: [33063 44775  5887 48525 41454 16434 49134 32286 29916 37990]
Selected labels: [1 9 7 6 2 4 8 7 7 2]
Classes covered: 7


## 6. Evaluation (Linear Probe)

In [8]:
def evaluate_selection(selected_indices, embeddings, labels, test_embeddings, test_labels):
    """Train a linear probe on selected examples and evaluate on test set."""
    sel_emb = embeddings[selected_indices]
    sel_lab = labels[selected_indices]
    clf = LogisticRegression(max_iter=1000)
    clf.fit(sel_emb, sel_lab)
    preds = clf.predict(test_embeddings)
    accuracy = accuracy_score(test_labels, preds)
    n_classes = len(np.unique(sel_lab))
    return {'accuracy': accuracy, 'n_classes_covered': n_classes}


results = evaluate_selection(selected_indices, embeddings, labels, test_embeddings, test_labels)
print(f"\n=== Original TypiClust Results ===")
print(f"Accuracy: {results['accuracy']:.4f}")
print(f"Classes covered: {results['n_classes_covered']}/10")


=== Original TypiClust Results ===
Accuracy: 0.4476
Classes covered: 7/10


## 7. Random Baseline Comparison

In [9]:
# Random baseline (30 seeds)
random_accs = []
for seed in range(30):
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(embeddings), size=budget, replace=False)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(embeddings[idx], labels[idx])
    preds = clf.predict(test_embeddings)
    random_accs.append(accuracy_score(test_labels, preds))

random_accs = np.array(random_accs)
print(f"Random Baseline: {random_accs.mean():.4f} +/- {random_accs.std():.4f}")
print(f"TypiClust: {results['accuracy']:.4f}")
print(f"Improvement: +{results['accuracy'] - random_accs.mean():.4f}")

Random Baseline: 0.3692 +/- 0.0675
TypiClust: 0.4476
Improvement: +0.0784
